# Regression Fundamentals: Predicting Housing Prices

In this tutorial, we explore the core ideas behind **regression** — the task of predicting a continuous numerical outcome from one or more input features. We will build intuition by working through a concrete scenario: **estimating the sale price of a home** given characteristics such as its square footage, number of bedrooms, age, and more.

Along the way we will cover:

| Topic | Why it matters |
|---|---|
| Exploratory Data Analysis | Understand structure & quirks before modeling |
| k-Nearest Neighbors (kNN) regression | A flexible, non-parametric baseline |
| Simple Linear Regression | The workhorse single-feature model |
| Multiple Linear Regression | Leveraging many features at once |
| Train / Test split | Honest evaluation of generalization |
| MSE & R² metrics | Quantifying prediction quality |
| Residual analysis | Diagnosing model assumptions |

Let's get started.

---
## 1. Setup and Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

# Plotting defaults
sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)
plt.rcParams["figure.dpi"] = 100

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("All imports successful.")

---
## 2. Data Loading and Exploration

We will simulate a realistic housing dataset inspired by the kinds of features found in the Ames Housing data. Simulating gives us full control over the ground-truth relationship, which is helpful for learning. The data generation process embeds a **known** relationship between features and price, so we can later check how well our models recover it.

In [ ]:
n_samples = 600

# --- Generate features ---
square_feet = np.random.normal(1800, 500, n_samples).clip(600, 5000).astype(int)
bedrooms = np.random.choice([2, 3, 3, 3, 4, 4, 5], size=n_samples)
bathrooms = np.random.choice([1, 1.5, 2, 2, 2.5, 3], size=n_samples)
lot_size = np.random.normal(8000, 3000, n_samples).clip(2000, 25000).astype(int)
year_built = np.random.randint(1950, 2023, n_samples)
garage_size = np.random.choice([0, 1, 1, 2, 2, 2, 3], size=n_samples)

# --- Ground-truth pricing model (with noise) ---
age = 2025 - year_built
price = (
    80 * square_feet
    + 15000 * bedrooms
    + 12000 * bathrooms
    + 3.5 * lot_size
    - 600 * age
    + 8000 * garage_size
    + np.random.normal(0, 25000, n_samples)   # irreducible noise
).clip(50000, None).astype(int)

# --- Assemble DataFrame ---
housing = pd.DataFrame({
    "square_feet": square_feet,
    "bedrooms": bedrooms,
    "bathrooms": bathrooms,
    "lot_size": lot_size,
    "year_built": year_built,
    "garage_size": garage_size,
    "price": price
})

print(f"Dataset shape: {housing.shape}")
housing.head(10)

### 2.1 Summary Statistics

In [ ]:
housing.describe().round(1)

### 2.2 Distribution of the Target Variable (Price)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(housing["price"], bins=30, edgecolor="white", color="steelblue")
axes[0].set_xlabel("Sale Price ($)")
axes[0].set_ylabel("Frequency")
axes[0].set_title("Distribution of Sale Price")

axes[1].boxplot(housing["price"], vert=True)
axes[1].set_ylabel("Sale Price ($)")
axes[1].set_title("Box Plot of Sale Price")

plt.tight_layout()
plt.show()

print(f"Median price: ${housing['price'].median():,.0f}")
print(f"Mean price:   ${housing['price'].mean():,.0f}")

### 2.3 Pairwise Relationships

A quick correlation heatmap tells us which features are most linearly associated with price.

In [ ]:
corr = housing.corr()

plt.figure(figsize=(9, 7))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, linewidths=0.5)
plt.title("Feature Correlation Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# Scatter plots of each feature vs price
features = ["square_feet", "bedrooms", "bathrooms", "lot_size", "year_built", "garage_size"]

fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, feat in zip(axes.ravel(), features):
    ax.scatter(housing[feat], housing["price"], alpha=0.3, s=12, color="steelblue")
    ax.set_xlabel(feat)
    ax.set_ylabel("price")
    ax.set_title(f"{feat} vs price  (r = {corr.loc[feat, 'price']:.2f})")

plt.suptitle("Individual Feature Relationships with Price", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

**Observations:**
- `square_feet` has the strongest positive correlation with price — larger homes cost more.
- `year_built` shows a modest positive trend (newer homes tend to be pricier).
- Features like `bedrooms` and `garage_size` are discrete, so the scatter plots look banded.

---
## 3. Single-Feature Regression: Square Feet → Price

We start with the simplest possible setup: predict price using **only** square footage. This lets us visualize predictions in 2-D and build intuition before adding complexity.

### 3.1 Train-Test Split

In [ ]:
# Perform a single train-test split on the full dataset.
# We'll use the same row indices for both single-feature and multi-feature models.
X_all = housing[features]
y = housing["price"]

X_train_all, X_test_all, y_train, y_test = train_test_split(
    X_all, y, test_size=0.2, random_state=RANDOM_STATE
)

# Single-feature versions (subset of columns, same rows)
X_train_s = X_train_all[["square_feet"]]
X_test_s = X_test_all[["square_feet"]]

print(f"Training samples: {len(X_train_s)}")
print(f"Test samples:     {len(X_test_s)}")

---
## 4. k-Nearest Neighbors Regression

kNN regression predicts the target for a new point by **averaging the targets of its k nearest training neighbors** (in feature space). The hyper-parameter *k* controls the balance between two competing forces:

| Small k | Large k |
|---|---|
| Flexible, wiggly fit | Smooth, stable fit |
| Low bias, high variance | Higher bias, low variance |
| Risk of overfitting | Risk of underfitting |

### 4.1 Fitting kNN with Several k Values

In [ ]:
k_values = [1, 3, 5, 10, 25, 50]

# We need to scale features for kNN (distance-based)
scaler_s = StandardScaler()
X_train_s_sc = scaler_s.fit_transform(X_train_s)
X_test_s_sc = scaler_s.transform(X_test_s)

knn_results = {}

for k in k_values:
    model = KNeighborsRegressor(n_neighbors=k)
    model.fit(X_train_s_sc, y_train)
    
    y_pred_train = model.predict(X_train_s_sc)
    y_pred_test = model.predict(X_test_s_sc)
    
    knn_results[k] = {
        "model": model,
        "train_mse": mean_squared_error(y_train, y_pred_train),
        "test_mse": mean_squared_error(y_test, y_pred_test),
        "train_r2": r2_score(y_train, y_pred_train),
        "test_r2": r2_score(y_test, y_pred_test),
    }

# Display results
results_df = pd.DataFrame(knn_results).T
results_df.index.name = "k"
results_df[["train_mse", "test_mse", "train_r2", "test_r2"]].round(2)

### 4.2 Visualizing the Bias-Variance Tradeoff

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 11))

# Generate a fine grid for smooth prediction curves
x_grid = np.linspace(X_single.min().values[0], X_single.max().values[0], 500).reshape(-1, 1)
x_grid_sc = scaler_s.transform(x_grid)

for ax, k in zip(axes.ravel(), k_values):
    model = knn_results[k]["model"]
    y_grid = model.predict(x_grid_sc)
    
    ax.scatter(X_train_s, y_train, alpha=0.2, s=10, label="Training data", color="steelblue")
    ax.plot(x_grid, y_grid, color="darkorange", linewidth=2, label=f"kNN (k={k})")
    
    ax.set_title(f"k = {k}   |   Test R² = {knn_results[k]['test_r2']:.3f}")
    ax.set_xlabel("Square Feet")
    ax.set_ylabel("Price ($)")
    ax.legend(loc="upper left", fontsize=9)

plt.suptitle("kNN Regression — Effect of k on Model Flexibility", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

**What to notice:**
- At **k=1** the prediction curve is extremely jagged — the model memorizes every training point (train R² = 1.0) but generalizes poorly.
- As **k increases**, the curve becomes smoother. The model starts capturing the overall trend rather than individual data points.
- Beyond a certain k the curve becomes *too* smooth and starts missing the true pattern (underfitting).

In [ ]:
# MSE vs k — classic bias-variance tradeoff plot
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ks = list(knn_results.keys())
train_mses = [knn_results[k]["train_mse"] for k in ks]
test_mses = [knn_results[k]["test_mse"] for k in ks]
train_r2s = [knn_results[k]["train_r2"] for k in ks]
test_r2s = [knn_results[k]["test_r2"] for k in ks]

ax1.plot(ks, train_mses, "o-", label="Train MSE", color="steelblue")
ax1.plot(ks, test_mses, "s--", label="Test MSE", color="darkorange")
ax1.set_xlabel("k (number of neighbors)")
ax1.set_ylabel("Mean Squared Error")
ax1.set_title("MSE vs k")
ax1.legend()

ax2.plot(ks, train_r2s, "o-", label="Train R²", color="steelblue")
ax2.plot(ks, test_r2s, "s--", label="Test R²", color="darkorange")
ax2.set_xlabel("k (number of neighbors)")
ax2.set_ylabel("R² Score")
ax2.set_title("R² vs k")
ax2.legend()

plt.suptitle("Bias-Variance Tradeoff in kNN Regression", fontsize=13)
plt.tight_layout()
plt.show()

best_k = ks[np.argmin(test_mses)]
print(f"Best k by test MSE: k = {best_k}")

---
## 5. Simple Linear Regression

Linear regression fits a straight line: $\hat{y} = \beta_0 + \beta_1 \cdot x$, choosing the intercept ($\beta_0$) and slope ($\beta_1$) that minimize the sum of squared residuals. Unlike kNN it makes a strong **parametric assumption** — that the relationship is linear.

### 5.1 Fit and Evaluate

In [ ]:
lr_simple = LinearRegression()
lr_simple.fit(X_train_s, y_train)

y_pred_train_lr = lr_simple.predict(X_train_s)
y_pred_test_lr = lr_simple.predict(X_test_s)

print("Simple Linear Regression: square_feet -> price")
print(f"  Intercept (β₀):  {lr_simple.intercept_:>12,.1f}")
print(f"  Slope (β₁):      {lr_simple.coef_[0]:>12,.2f}  ($ per sq ft)")
print()
print(f"  Train MSE: {mean_squared_error(y_train, y_pred_train_lr):>14,.0f}")
print(f"  Test  MSE: {mean_squared_error(y_test, y_pred_test_lr):>14,.0f}")
print(f"  Train R²:  {r2_score(y_train, y_pred_train_lr):>10.4f}")
print(f"  Test  R²:  {r2_score(y_test, y_pred_test_lr):>10.4f}")

### 5.2 Visualizing the Fit

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Left: regression line on training data
ax1.scatter(X_train_s, y_train, alpha=0.25, s=12, color="steelblue", label="Train")
ax1.scatter(X_test_s, y_test, alpha=0.25, s=12, color="seagreen", label="Test")
x_line = np.linspace(X_single.min().values[0], X_single.max().values[0], 100).reshape(-1, 1)
ax1.plot(x_line, lr_simple.predict(x_line), color="crimson", linewidth=2, label="Linear fit")
ax1.set_xlabel("Square Feet")
ax1.set_ylabel("Price ($)")
ax1.set_title("Simple Linear Regression")
ax1.legend()

# Right: predicted vs actual
ax2.scatter(y_test, y_pred_test_lr, alpha=0.4, s=15, color="steelblue")
limits = [y_test.min(), y_test.max()]
ax2.plot(limits, limits, "--", color="gray", linewidth=1, label="Perfect prediction")
ax2.set_xlabel("Actual Price ($)")
ax2.set_ylabel("Predicted Price ($)")
ax2.set_title("Predicted vs Actual (Test Set)")
ax2.legend()

plt.tight_layout()
plt.show()

The linear fit captures the dominant trend well, but the spread around the line reminds us that square footage alone does not explain all the variation in price. We need more features.

---
## 6. Multiple Linear Regression

Now we include **all six features**. The model becomes:

$$\hat{y} = \beta_0 + \beta_1 \cdot \text{sqft} + \beta_2 \cdot \text{beds} + \beta_3 \cdot \text{baths} + \beta_4 \cdot \text{lot} + \beta_5 \cdot \text{year} + \beta_6 \cdot \text{garage}$$

### 6.1 Prepare Data

In [ ]:
# Multi-feature versions use the same train-test split from earlier
X_train_m = X_train_all[features]
X_test_m = X_test_all[features]

print(f"Features: {list(X_train_m.columns)}")
print(f"Training: {X_train_m.shape}  |  Test: {X_test_m.shape}")

### 6.2 Fit and Evaluate

In [ ]:
lr_multi = LinearRegression()
lr_multi.fit(X_train_m, y_train)

y_pred_train_multi = lr_multi.predict(X_train_m)
y_pred_test_multi = lr_multi.predict(X_test_m)

print("Multiple Linear Regression Coefficients:")
print(f"  {'Intercept':<14} {lr_multi.intercept_:>12,.1f}")
for name, coef in zip(features, lr_multi.coef_):
    print(f"  {name:<14} {coef:>12,.2f}")

print()
print(f"  Train MSE: {mean_squared_error(y_train, y_pred_train_multi):>14,.0f}")
print(f"  Test  MSE: {mean_squared_error(y_test, y_pred_test_multi):>14,.0f}")
print(f"  Train R²:  {r2_score(y_train, y_pred_train_multi):>10.4f}")
print(f"  Test  R²:  {r2_score(y_test, y_pred_test_multi):>10.4f}")

### 6.3 Coefficient Interpretation

Each coefficient tells us: *holding all other features constant, how much does a one-unit increase in this feature change the predicted price?*

In [ ]:
coef_df = pd.DataFrame({
    "Feature": features,
    "Coefficient": lr_multi.coef_
}).sort_values("Coefficient", key=abs, ascending=True)

plt.figure(figsize=(8, 5))
colors = ["steelblue" if c > 0 else "salmon" for c in coef_df["Coefficient"]]
plt.barh(coef_df["Feature"], coef_df["Coefficient"], color=colors, edgecolor="white")
plt.xlabel("Coefficient Value ($ per unit)")
plt.title("Multiple Regression — Feature Coefficients")
plt.tight_layout()
plt.show()

### 6.4 Predicted vs Actual

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Training set
ax1.scatter(y_train, y_pred_train_multi, alpha=0.3, s=12, color="steelblue")
lims = [min(y_train.min(), y_pred_train_multi.min()),
        max(y_train.max(), y_pred_train_multi.max())]
ax1.plot(lims, lims, "--", color="gray")
ax1.set_xlabel("Actual Price ($)")
ax1.set_ylabel("Predicted Price ($)")
ax1.set_title(f"Training Set  (R² = {r2_score(y_train, y_pred_train_multi):.3f})")

# Test set
ax2.scatter(y_test, y_pred_test_multi, alpha=0.3, s=12, color="darkorange")
lims = [min(y_test.min(), y_pred_test_multi.min()),
        max(y_test.max(), y_pred_test_multi.max())]
ax2.plot(lims, lims, "--", color="gray")
ax2.set_xlabel("Actual Price ($)")
ax2.set_ylabel("Predicted Price ($)")
ax2.set_title(f"Test Set  (R² = {r2_score(y_test, y_pred_test_multi):.3f})")

plt.suptitle("Multiple Linear Regression — Predicted vs Actual", fontsize=13)
plt.tight_layout()
plt.show()

---
## 7. Residual Analysis and Model Diagnostics

Residuals are the differences between actual and predicted values: $e_i = y_i - \hat{y}_i$.

For a well-specified linear model we expect:
1. **Residuals centered around zero** — no systematic bias.
2. **Constant variance** (homoscedasticity) — the spread of residuals should not depend on the predicted value.
3. **Approximately normal distribution** of residuals.

### 7.1 Residual Plots for Multiple Regression

In [ ]:
residuals_train = y_train - y_pred_train_multi
residuals_test = y_test - y_pred_test_multi

fig, axes = plt.subplots(2, 2, figsize=(14, 11))

# 1. Residuals vs Predicted
axes[0, 0].scatter(y_pred_test_multi, residuals_test, alpha=0.35, s=12, color="steelblue")
axes[0, 0].axhline(0, color="crimson", linestyle="--", linewidth=1)
axes[0, 0].set_xlabel("Predicted Price ($)")
axes[0, 0].set_ylabel("Residual ($)")
axes[0, 0].set_title("Residuals vs Predicted Values")

# 2. Histogram of residuals
axes[0, 1].hist(residuals_test, bins=30, edgecolor="white", color="steelblue", density=True)
# Overlay a normal curve
xr = np.linspace(residuals_test.min(), residuals_test.max(), 200)
axes[0, 1].plot(xr, stats.norm.pdf(xr, residuals_test.mean(), residuals_test.std()),
                color="crimson", linewidth=2, label="Normal fit")
axes[0, 1].set_xlabel("Residual ($)")
axes[0, 1].set_ylabel("Density")
axes[0, 1].set_title("Distribution of Residuals")
axes[0, 1].legend()

# 3. Q-Q plot
stats.probplot(residuals_test, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title("Normal Q-Q Plot of Residuals")
axes[1, 0].get_lines()[0].set_color("steelblue")
axes[1, 0].get_lines()[0].set_alpha(0.5)

# 4. Residuals vs each feature (use square_feet as example)
axes[1, 1].scatter(X_test_m["square_feet"], residuals_test, alpha=0.35, s=12, color="steelblue")
axes[1, 1].axhline(0, color="crimson", linestyle="--", linewidth=1)
axes[1, 1].set_xlabel("Square Feet")
axes[1, 1].set_ylabel("Residual ($)")
axes[1, 1].set_title("Residuals vs Square Feet")

plt.suptitle("Model Diagnostics — Multiple Linear Regression", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

**Interpreting the diagnostics:**
- **Residuals vs Predicted**: Ideally a random cloud around zero. Any funnel shape would indicate heteroscedasticity; any curve would suggest a missing non-linear term.
- **Histogram / Q-Q plot**: Residuals should be roughly bell-shaped. Heavy tails or skewness may point to outliers or a mis-specified model.
- **Residuals vs Feature**: No pattern means the linear assumption is reasonable for that feature.

### 7.2 Residual Comparison: Single-Feature kNN vs Multi-Feature Linear Regression

**Note:** This is deliberately an apples-to-oranges comparison — the kNN model uses only square footage, while the linear regression uses all six features. The point is to see how adding more information (features) tightens the residual distribution.

In [ ]:
best_knn = knn_results[best_k]["model"]
y_pred_knn_test = best_knn.predict(X_test_s_sc)
residuals_knn = y_test - y_pred_knn_test

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.scatter(y_pred_knn_test, residuals_knn, alpha=0.35, s=12, color="darkorange")
ax1.axhline(0, color="crimson", linestyle="--")
ax1.set_xlabel("Predicted Price ($)")
ax1.set_ylabel("Residual ($)")
ax1.set_title(f"kNN (k={best_k}) Residuals vs Predicted")

ax2.hist(residuals_knn, bins=30, edgecolor="white", color="darkorange", alpha=0.7, label=f"kNN k={best_k}")
ax2.hist(residuals_test, bins=30, edgecolor="white", color="steelblue", alpha=0.5, label="Multi LR")
ax2.set_xlabel("Residual ($)")
ax2.set_ylabel("Frequency")
ax2.set_title("Residual Distributions Compared")
ax2.legend()

plt.tight_layout()
plt.show()

---
## 8. Model Comparison Summary

Let's also fit a multi-feature kNN and then gather all results into one table.

In [ ]:
# Multi-feature kNN
scaler_m = StandardScaler()
X_train_m_sc = scaler_m.fit_transform(X_train_m)
X_test_m_sc = scaler_m.transform(X_test_m)

knn_multi = KNeighborsRegressor(n_neighbors=best_k)
knn_multi.fit(X_train_m_sc, y_train)

y_pred_knn_multi_train = knn_multi.predict(X_train_m_sc)
y_pred_knn_multi_test = knn_multi.predict(X_test_m_sc)

In [ ]:
# Build comparison table — all models use the same y_train / y_test
comparison = pd.DataFrame({
    "Model": [
        f"kNN (k={best_k}, sqft only)",
        "Simple Linear Reg (sqft only)",
        f"kNN (k={best_k}, all features)",
        "Multiple Linear Reg (all features)"
    ],
    "Train MSE": [
        mean_squared_error(y_train, best_knn.predict(X_train_s_sc)),
        mean_squared_error(y_train, y_pred_train_lr),
        mean_squared_error(y_train, y_pred_knn_multi_train),
        mean_squared_error(y_train, y_pred_train_multi)
    ],
    "Test MSE": [
        mean_squared_error(y_test, y_pred_knn_test),
        mean_squared_error(y_test, y_pred_test_lr),
        mean_squared_error(y_test, y_pred_knn_multi_test),
        mean_squared_error(y_test, y_pred_test_multi)
    ],
    "Train R²": [
        r2_score(y_train, best_knn.predict(X_train_s_sc)),
        r2_score(y_train, y_pred_train_lr),
        r2_score(y_train, y_pred_knn_multi_train),
        r2_score(y_train, y_pred_train_multi)
    ],
    "Test R²": [
        r2_score(y_test, y_pred_knn_test),
        r2_score(y_test, y_pred_test_lr),
        r2_score(y_test, y_pred_knn_multi_test),
        r2_score(y_test, y_pred_test_multi)
    ]
})

comparison["Test RMSE"] = np.sqrt(comparison["Test MSE"])
comparison = comparison[["Model", "Train MSE", "Test MSE", "Test RMSE", "Train R²", "Test R²"]]

print("=" * 90)
print("MODEL COMPARISON")
print("=" * 90)
comparison.style.format({
    "Train MSE": "{:,.0f}",
    "Test MSE": "{:,.0f}",
    "Test RMSE": "${:,.0f}",
    "Train R²": "{:.4f}",
    "Test R²": "{:.4f}"
})

In [ ]:
# Visual comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

colors = ["#4e79a7", "#f28e2b", "#59a14f", "#e15759"]
models = comparison["Model"]

ax1.barh(models, comparison["Test RMSE"], color=colors, edgecolor="white")
ax1.set_xlabel("Test RMSE ($)")
ax1.set_title("Test RMSE (lower is better)")
for i, v in enumerate(comparison["Test RMSE"]):
    ax1.text(v + 500, i, f"${v:,.0f}", va="center", fontsize=10)

ax2.barh(models, comparison["Test R²"], color=colors, edgecolor="white")
ax2.set_xlabel("Test R²")
ax2.set_title("Test R² (higher is better)")
ax2.set_xlim(0, 1.05)
for i, v in enumerate(comparison["Test R²"]):
    ax2.text(v + 0.01, i, f"{v:.3f}", va="center", fontsize=10)

plt.suptitle("Model Performance Comparison", fontsize=14)
plt.tight_layout()
plt.show()

---
## 9. Wrap-Up and Key Takeaways

### What we learned

1. **Exploratory Data Analysis** is essential before modeling. Understanding feature distributions, correlations, and potential outliers guides every subsequent decision.

2. **kNN Regression** is a non-parametric method that makes no assumptions about the functional form of the relationship. Its flexibility is controlled by *k*:
   - Small *k* → complex model → risk of overfitting
   - Large *k* → simple model → risk of underfitting
   - This is the **bias-variance tradeoff** in action.

3. **Simple Linear Regression** assumes a straight-line relationship between one predictor and the target. It is interpretable (slope = marginal effect) but limited when multiple factors influence the outcome.

4. **Multiple Linear Regression** extends the idea to many features simultaneously. The coefficients tell us the effect of each feature while controlling for the others. In our experiment, it achieved the best test-set performance because the true data-generating process was itself linear.

5. **Train-test splits** are critical for honest evaluation. A model that looks great on training data may generalize poorly (as we saw with kNN at k=1).

6. **MSE and R²** are complementary metrics. MSE gives the error in the same (squared) units as the target; R² tells us the proportion of variance explained.

7. **Residual analysis** checks whether the model's assumptions hold. Patterns in residual plots signal opportunities to improve the model.

### Next steps to explore
- **Cross-validation** for more robust model selection (rather than a single train-test split)
- **Regularization** (Ridge, Lasso) to handle multicollinearity and high-dimensional features
- **Polynomial features** to capture non-linear relationships within a linear regression framework
- **Real-world datasets** (e.g., Ames Housing on Kaggle) where the true relationship is unknown